In [7]:
from datasets import load_dataset
from rouge_score import rouge_scorer
import nltk
from extractive_functions import Extractive_Summarizer  # your custom function

# Download NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

# %%
# Load CNN/DailyMail dataset (3.0.0 is the latest clean split)
dataset = load_dataset("cnn_dailymail", "3.0.0", split="test")

# Use a small subset for testing
num_samples = 200
sample_dataset = dataset.select(range(num_samples))

# %%
# Initialize ROUGE scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# Initialize score storage
all_rouge1 = []
all_rouge2 = []
all_rougeL = []

# %%
# Process each sample
for i, example in enumerate(sample_dataset):
    document_text = example['article']
    reference_summary = example['highlights']  # CNN/DM has extractive-friendly summaries

    try:
        # Generate extractive summary (your function)
        generated_summary, _ = Extractive_Summarizer(
            document_text, ratio=0.2, selectedOptionValue="very_short"
        )
    except Exception as e:
        print(f"Skipping sample {i} due to summarization error: {e}")
        continue

    # Compute ROUGE
    scores = scorer.score(reference_summary, generated_summary)

    # Store F1 scores (commonly reported in summarization)
    all_rouge1.append(scores['rouge1'].fmeasure)
    all_rouge2.append(scores['rouge2'].fmeasure)
    all_rougeL.append(scores['rougeL'].fmeasure)

    if (i + 1) % 20 == 0:
        print(f"Processed {i+1}/{num_samples} samples...")

# %%
# Calculate averages
avg_rouge1 = sum(all_rouge1) / len(all_rouge1) if all_rouge1 else 0
avg_rouge2 = sum(all_rouge2) / len(all_rouge2) if all_rouge2 else 0
avg_rougeL = sum(all_rougeL) / len(all_rougeL) if all_rougeL else 0

print("==== Final Results on CNN/DailyMail Subset ====")
print(f"Average ROUGE-1 F1: {avg_rouge1:.4f}")
print(f"Average ROUGE-2 F1: {avg_rouge2:.4f}")
print(f"Average ROUGE-L F1: {avg_rougeL:.4f}")

# ROUGE-1 F1 ≈ 0.40–0.43
# ROUGE-2 F1 ≈ 0.17–0.20
# ROUGE-L F1 ≈ 0.36


# ==== Final Results on CNN/DailyMail Subset ====
# Average ROUGE-1 F1: 0.2124
# Average ROUGE-2 F1: 0.0809
# Average ROUGE-L F1: 0.1443

Processed 20/200 samples...
Processed 40/200 samples...
Processed 60/200 samples...
Processed 80/200 samples...
Processed 100/200 samples...
Processed 120/200 samples...
Processed 140/200 samples...
Processed 160/200 samples...
Processed 180/200 samples...
Processed 200/200 samples...
==== Final Results on CNN/DailyMail Subset ====
Average ROUGE-1 F1: 0.2803
Average ROUGE-2 F1: 0.0903
Average ROUGE-L F1: 0.1876


In [4]:
from datasets import load_dataset
from rouge_score import rouge_scorer
import nltk
from extractive_functions import Extractive_Summarizer  # your custom function

# Download NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

# %%
# Load BillSum dataset (use the California subset for smaller size)
dataset = load_dataset("billsum", split="ca_test")

# Use a smaller subset for quick testing
num_samples = 100
sample_dataset = dataset.select(range(num_samples))

# %%
# Initialize ROUGE scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# Store results
all_rouge1, all_rouge2, all_rougeL = [], [], []

# %%
# Process each sample
for i, example in enumerate(sample_dataset):
    document_text = example['text']            # full bill text
    reference_summary = example['summary']     # human-written summary

    try:
        generated_summary, _ = Extractive_Summarizer(
            document_text, ratio=0.15, selectedOptionValue="dummy"
        )
    except Exception as e:
        print(f"Skipping sample {i} due to summarization error: {e}")
        continue

    # Compute ROUGE
    scores = scorer.score(reference_summary, generated_summary)

    all_rouge1.append(scores['rouge1'].fmeasure)
    all_rouge2.append(scores['rouge2'].fmeasure)
    all_rougeL.append(scores['rougeL'].fmeasure)

    if (i + 1) % 10 == 0:
        print(f"Processed {i+1}/{num_samples} samples...")

# %%
# Final averages
avg_rouge1 = sum(all_rouge1) / len(all_rouge1) if all_rouge1 else 0
avg_rouge2 = sum(all_rouge2) / len(all_rouge2) if all_rouge2 else 0
avg_rougeL = sum(all_rougeL) / len(all_rougeL) if all_rougeL else 0

print("==== Final Results on BillSum (CA Test Subset) ====")
print(f"Average ROUGE-1 F1: {avg_rouge1:.4f}")
print(f"Average ROUGE-2 F1: {avg_rouge2:.4f}")
print(f"Average ROUGE-L F1: {avg_rougeL:.4f}")


README.md: 0.00B [00:00, ?B/s]

c:\Users\Dell\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dell\.cache\huggingface\hub\datasets--billsum. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better perfor

train-00000-of-00001.parquet:   0%|          | 0.00/91.8M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


test-00000-of-00001.parquet:   0%|          | 0.00/15.8M [00:00<?, ?B/s]

...

ca_test-00000-of-00001.parquet:   0%|          | 0.00/6.12M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18949 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3269 [00:00<?, ? examples/s]

Generating ca_test split:   0%|          | 0/1237 [00:00<?, ? examples/s]

Processed 10/100 samples...
Processed 20/100 samples...
Processed 30/100 samples...
Processed 40/100 samples...
Processed 50/100 samples...
Processed 60/100 samples...
Processed 70/100 samples...
Processed 80/100 samples...
Processed 90/100 samples...
Processed 100/100 samples...
==== Final Results on BillSum (CA Test Subset) ====
Average ROUGE-1 F1: 0.1897
Average ROUGE-2 F1: 0.0835
Average ROUGE-L F1: 0.1308
